# Introduction to Apache Spark and PySpark

## 📚 Learning Objectives
By the end of this notebook, you will:
- Understand what Apache Spark is and why it's important
- Learn the difference between Spark and Hadoop MapReduce
- Understand Spark's architecture (Driver, Executors, Cluster Manager)
- Learn about Spark components (Core, SQL, Streaming, MLlib)
- Create your first PySpark session

---

## 🎯 What is Apache Spark?

**Apache Spark** is a unified analytics engine for large-scale data processing.

### Key Features:
1. **Speed**: 100x faster than Hadoop MapReduce (in-memory processing)
2. **Easy to Use**: High-level APIs in Python, Scala, Java, R
3. **General Purpose**: Batch, streaming, ML, graph processing
4. **Runs Everywhere**: Hadoop, Kubernetes, standalone, cloud

### Why PySpark?
- Python is easy to learn and widely used
- Access to Python libraries (pandas, numpy, scikit-learn)
- Great for data science and machine learning
- Industry standard for big data processing

---

## ⚖️ Spark vs Hadoop MapReduce

| Feature | Hadoop MapReduce | Apache Spark |
|---------|-----------------|-------------|
| **Processing** | Disk-based | In-memory |
| **Speed** | Slower | 100x faster |
| **Ease of Use** | Complex | Easy (high-level APIs) |
| **Real-time** | ❌ No | ✅ Yes |
| **Machine Learning** | Limited | ✅ MLlib |
| **Iterative Processing** | Slow | Fast |

### Trade-off:
- **Spark**: Faster but requires more memory
- **MapReduce**: Slower but works with limited memory

---

## 🏗️ Spark Architecture

```
┌─────────────────────────────────────────┐
│          Driver Program                 │
│     (SparkContext/SparkSession)         │
│  - Converts code to tasks               │
│  - Schedules tasks on executors         │
└──────────────┬──────────────────────────┘
               │
               ▼
┌──────────────────────────────────────────┐
│        Cluster Manager                   │
│  (YARN, Mesos, Kubernetes, Standalone)   │
│  - Allocates resources                   │
└──────────────┬───────────────────────────┘
               │
       ┌───────┴────────┬────────┐
       ▼                ▼        ▼
  ┌─────────┐    ┌─────────┐  ┌─────────┐
  │Executor │    │Executor │  │Executor │
  │  Task   │    │  Task   │  │  Task   │
  │  Task   │    │  Task   │  │  Task   │
  │ Cache   │    │ Cache   │  │ Cache   │
  └─────────┘    └─────────┘  └─────────┘
```

### Components:
1. **Driver**: Main program that creates SparkContext
2. **Cluster Manager**: Allocates resources (YARN, Mesos, K8s)
3. **Executors**: Worker nodes that run tasks and cache data

---

## 🧩 Spark Components

```
┌─────────────────────────────────────────┐
│           Spark SQL                     │
│    (DataFrames, Datasets, SQL)          │
├─────────────────────────────────────────┤
│  Spark        │  Spark      │  GraphX   │
│  Streaming    │  MLlib      │  (Graphs) │
├───────────────┴─────────────┴───────────┤
│           Spark Core                    │
│   (RDD, Task Scheduling, Memory Mgmt)   │
└─────────────────────────────────────────┘
```

1. **Spark Core**: RDDs, task scheduling
2. **Spark SQL**: Structured data (DataFrames)
3. **Spark Streaming**: Real-time processing
4. **MLlib**: Machine learning
5. **GraphX**: Graph processing

---

## 💻 Practical: Creating Your First Spark Session

Let's create a SparkSession and run some basic commands!

In [ ]:
# Import necessary libraries
from pyspark.sql import SparkSession

# Create SparkSession
spark = SparkSession.builder \
    .appName("Intro to Spark") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("✅ SparkSession created successfully!")
print(f"Spark Version: {spark.version}")
print(f"App Name: {spark.sparkContext.appName}")

### Understanding the Code:

- **`.appName()`**: Name of your Spark application
- **`.master("local[*]")`**: Run locally with all available cores
  - `local[1]` = 1 core
  - `local[4]` = 4 cores
  - `local[*]` = all available cores
- **`.config()`**: Set Spark configurations
- **`.getOrCreate()`**: Get existing session or create new one

In [ ]:
# Check Spark context information
sc = spark.sparkContext
print(f"Master: {sc.master}")
print(f"App ID: {sc.applicationId}")
print(f"Default Parallelism: {sc.defaultParallelism}")

## 🎯 Hello World with PySpark

Let's create a simple example to understand how Spark works.

In [ ]:
# Example 1: Creating a simple DataFrame
data = [("Alice", 25), ("Bob", 30), ("Charlie", 35)]
columns = ["Name", "Age"]

df = spark.createDataFrame(data, columns)
df.show()

print(f"\nDataFrame has {df.count()} rows")
print(f"Schema: {df.schema}")

In [ ]:
# Example 2: Simple transformations
from pyspark.sql.functions import col

# Filter people older than 25
filtered_df = df.filter(col("Age") > 25)
print("People older than 25:")
filtered_df.show()

# Add a new column
df_with_category = df.withColumn(
    "Category",
    when(col("Age") < 30, "Young").otherwise("Senior")
)
print("\nWith age category:")
df_with_category.show()

## 🔍 Spark UI - Monitoring Your Jobs

While your Spark session is running, you can access the Spark UI at:
**http://localhost:4040**

The UI shows:
- Jobs and stages
- Storage (cached RDDs/DataFrames)
- Environment configuration
- Executors information

### 💡 Best Practice:
Always monitor your jobs using Spark UI to understand:
- Where time is spent
- Data skew issues
- Memory usage
- Failed tasks

## 📝 Key Concepts Summary

### 1. Lazy Evaluation
- Transformations are **lazy** (not executed immediately)
- Actions **trigger** execution
- Builds an execution plan (DAG)

```python
df1 = df.filter(col("Age") > 25)  # Lazy - not executed
df2 = df1.select("Name")          # Lazy - not executed
df2.show()                         # Action - executes entire chain
```

### 2. Transformations vs Actions

**Transformations** (return new DataFrame/RDD):
- `select()`, `filter()`, `groupBy()`, `join()`
- Lazy execution

**Actions** (return results):
- `show()`, `count()`, `collect()`, `save()`
- Trigger execution

### 3. DAG (Directed Acyclic Graph)
- Spark builds an execution plan
- Optimizes the plan before execution
- Enables fault tolerance

In [ ]:
# Example: Understanding lazy evaluation
print("Creating transformations...")
lazy_df = df.filter(col("Age") > 25) \
           .select("Name") \
           .orderBy("Name")
print("Transformations created (not executed yet)")

print("\nTriggering action...")
lazy_df.show()  # This triggers execution of all transformations
print("Action completed!")

## 🎯 Practice Exercises

Try these exercises to reinforce your learning:

### Exercise 1: Create Your Own DataFrame
Create a DataFrame with information about 5 cities including:
- City name
- Population
- Country

In [ ]:
# Your code here
# Solution in solutions notebook


### Exercise 2: Filter and Transform
Using the cities DataFrame:
1. Filter cities with population > 1 million
2. Add a column indicating if it's a "Megacity"
3. Show the results

In [ ]:
# Your code here


## 🎓 Key Takeaways

✅ Spark is a unified analytics engine for big data processing

✅ It's 100x faster than MapReduce due to in-memory processing

✅ Spark has Driver, Cluster Manager, and Executors

✅ Transformations are lazy, actions trigger execution

✅ Always use SparkSession as the entry point

✅ Monitor jobs using Spark UI (http://localhost:4040)

---

## 📚 Next Steps

Continue to: **02_rdd_basics.ipynb**

You'll learn about:
- What are RDDs?
- Creating RDDs
- Basic transformations and actions
- RDD lineage

In [ ]:
# Always stop your SparkSession when done
spark.stop()
print("✅ SparkSession stopped")